In [ ]:
!nvidia-smi

In [ ]:
from keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.layers import Dense, Input, LSTM, Embedding, Dropout, Activation
from keras.layers import Bidirectional, GlobalMaxPool1D
from keras.models import Model
from keras import initializers, regularizers, constraints, optimizers, layers
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [ ]:
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/00359/NewsAggregatorDataset.zip
!unzip NewsAggregatorDataset.zip

In [ ]:
# ???????????????????????????????????????
!sed -e 's/"/'\''/g' ./newsCorpora.csv > ./newsCorpora_re.csv

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# ??????
df = pd.read_csv('./newsCorpora_re.csv', header=None, sep='\t', names=['ID', 'TITLE', 'URL', 'PUBLISHER', 'CATEGORY', 'STORY', 'HOSTNAME', 'TIMESTAMP'])

# ??????
df = df.loc[df['PUBLISHER'].isin(['Reuters', 'Huffington Post', 'Businessweek', 'Contactmusic.com', 'Daily Mail']), ['TITLE', 'CATEGORY']]

# ??????
train, valid_test = train_test_split(df, test_size=0.2, shuffle=True, random_state=123, stratify=df['CATEGORY'])
valid, test = train_test_split(valid_test, test_size=0.5, shuffle=True, random_state=123, stratify=valid_test['CATEGORY'])
train.reset_index(drop=True, inplace=True)
valid.reset_index(drop=True, inplace=True)
test.reset_index(drop=True, inplace=True)

print(train.head())

In [ ]:
# GloVe??????
!wget https://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

In [ ]:
!ls

In [ ]:
EMBEDDING_FILE='./glove.6B.300d.txt'

In [ ]:
embed_size = 300 # how big is each word vector
max_features = 14724 # how many unique words to use (i.e num rows in embedding vector)
maxlen = 100 # max number of words in a comment to use

In [ ]:
label_to_num = {"b": 0, "e": 1, "m": 2, "t": 3}

In [ ]:
list_sentences_train = train["TITLE"].values
list_sentences_valid = valid["TITLE"].values
list_sentences_test = test["TITLE"].values
ohe = OneHotEncoder()
y_tr = ohe.fit_transform(np.array(train["CATEGORY"]).reshape(-1,1)).toarray()

In [ ]:
y_tr

In [ ]:
tokenizer = Tokenizer(num_words=max_features)
tokenizer.fit_on_texts(list(list_sentences_train))
list_tokenized_train = tokenizer.texts_to_sequences(list_sentences_train)
list_tokenized_valid = tokenizer.texts_to_sequences(list_sentences_valid)
list_tokenized_test = tokenizer.texts_to_sequences(list_sentences_test)
X_tr = pad_sequences(list_tokenized_train, maxlen=maxlen)
X_vl = pad_sequences(list_tokenized_valid, maxlen=maxlen)
X_te = pad_sequences(list_tokenized_test, maxlen=maxlen)

In [ ]:
# Read the glove word vectors (space delimited strings) into a dictionary from word->vector.

def get_coefs(word,*arr): return word, np.asarray(arr, dtype='float32')
embeddings_index = dict(get_coefs(*o.strip().split()) for o in open(EMBEDDING_FILE))

In [ ]:
'''
Use these vectors to create our embedding matrix, 
with random initialization for words that aren't in GloVe. 
We'll use the same mean and stdev of embeddings the GloVe has 
when generating the random init.
'''
all_embs = np.stack(embeddings_index.values())
emb_mean,emb_std = all_embs.mean(), all_embs.std()
emb_mean,emb_std

In [ ]:
word_index = tokenizer.word_index
nb_words = min(max_features, len(word_index))
embedding_matrix = np.random.normal(emb_mean, emb_std, (nb_words, embed_size))
for word, i in word_index.items():
    if i >= max_features: continue
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None: embedding_matrix[i] = embedding_vector

In [ ]:
'''
Simple bidirectional LSTM with two fully connected layers. 
We add some dropout to the LSTM since even 2 epochs is enough to overfit.
'''
inp = Input(shape=(maxlen,))
x = Embedding(max_features, embed_size, weights=[embedding_matrix])(inp)
x = Bidirectional(LSTM(300, return_sequences=True, dropout=0.1, recurrent_dropout=0.1))(x)
x = GlobalMaxPool1D()(x)
x = Dense(300, activation="relu")(x)
x = Dropout(0.1)(x)
x = Dense(4, activation="sigmoid")(x)
model = Model(inputs=inp, outputs=x)
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

In [ ]:
model.fit(X_tr, y_tr, batch_size=32, epochs=2, validation_split=0)

In [ ]:
y_te = ohe.transform(np.array(test["CATEGORY"]).reshape(-1,1)).toarray()
results = model.evaluate(X_te, y_te, batch_size=128)

In [ ]:
y_te = y_te.argmax(axis=1)

In [ ]:
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score
y_pred = model.predict(X_te)
y_pred_argmax = y_pred.argmax(axis=1)
print('{:.4f}'.format(accuracy_score(y_te, y_pred_argmax)))
y_te = ohe.transform(np.array(test["CATEGORY"]).reshape(-1,1)).toarray()
print('{:.4f}'.format(roc_auc_score(y_te, y_pred, multi_class='ovo')))
print('{:.4f}'.format(log_loss(y_te, y_pred)))

In [ ]:
y_vl = y_vl.argmax(axis=1)
y_pred = model.predict(X_vl)
y_pred_argmax = y_pred.argmax(axis=1)
print('{:.4f}'.format(accuracy_score(y_vl, y_pred_argmax)))
y_vl = ohe.transform(np.array(valid["CATEGORY"]).reshape(-1,1)).toarray()
print('{:.4f}'.format(roc_auc_score(y_vl, y_pred, multi_class='ovo')))
print('{:.4f}'.format(log_loss(y_vl, y_pred)))